# Comparaison de nos 3 modèles

On va faire un test pour comparer nos 3 modèles.


In [41]:
#Chargement de nos données et modèles

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer

df = pd.read_csv("../../datasets/clean/perfumes_clean.csv")
colonnes_texte = df.select_dtypes(include=["object", "string"]).columns
df[colonnes_texte] = df[colonnes_texte].fillna("")


mots_inutiles = ["olfactory", "family", "character", "notes", "seasons", "time", "day", "heart", "base"]

def tokeniser(texte):
    mots = texte.lower().replace(",", "").replace(".", "").replace(":", "").split()
    return [m for m in mots if m not in ENGLISH_STOP_WORDS and m not in mots_inutiles]

corpus = df["text_full"].apply(tokeniser)
w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=1, workers=1, sg=1, epochs=100, seed=42)
vecteurs_w2v = np.array([np.mean([w2v.wv[m] for m in mots], axis=0) for mots in corpus])


modele = SentenceTransformer("all-MiniLM-L6-v2")
emb_ambiance = modele.encode((df["description"] + " Character: " + df["character"]).tolist())
emb_notes = modele.encode(("Olfactory family: " + df["family"] + " " + df["secondary_family"] + ". Notes: " + df["notes"]).tolist())
emb_contexte = modele.encode(("Season: " + df["seasons"] + ". Time of day: " + df["times_of_day"] + ". Concentration: " + df["concentration"]).tolist())


vecteurs_texte = modele.encode(df["text_full"].tolist())
vecteurs_character = modele.encode(("A " + df["character"].str.replace("; ", ", ") + " scent.").tolist())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4477.52it/s]


In [45]:
def sans_notes_detestees(profil):
    masque = pd.Series(True, index=df.index)
    for note in profil["naime_pas"]:
        masque &= ~df["notes"].str.contains(note, case=False, regex=False)
    return masque

def recommander_w2v(profil):
    mots = [m for m in tokeniser(profil["description"]) if m in w2v.wv]
    if not mots:
        return None
    scores = cosine_similarity(np.mean([w2v.wv[m] for m in mots], axis=0).reshape(1, -1), vecteurs_w2v)[0]
    masque = sans_notes_detestees(profil) & (df[[f"time_{m}"for m in profil["moments"]]].max(axis=1) == 1)
    return df.loc[masque].assign(score=scores[masque]).sort_values("score", ascending=False).head(3)

def recommander_st(profil):
    score = (0.45 * cosine_similarity(modele.encode([profil["description"] + ". " + profil["emotions"]]), emb_ambiance)[0]
             + 0.35 * cosine_similarity(modele.encode([profil["description"]]), emb_notes)[0]
             + 0.20 * cosine_similarity(modele.encode([f"Perfect for the {' and '.join(profil['moments'])} in {' and '.join(profil['saisons'])}"]), emb_contexte)[0])
    score = 0.85 * score + 0.15 * (1 - np.abs(df["intensity"].fillna(3.0).values - profil["intensite"]) / 4)
    masque = sans_notes_detestees(profil) & (df[[f"time_{m}" for m in profil["moments"]]].max(axis=1) == 1)
    return df.loc[masque].assign(score=score[masque]).sort_values("score", ascending=False).head(3)

def normaliser(x):
    return np.full(len(x), 0.5) if x.max() == x.min() else (x - x.min()) / (x.max() - x.min())

def recommander_hybride(profil):
    c = df[sans_notes_detestees(profil)].copy()
    s_sem = normaliser(cosine_similarity(modele.encode([profil["description"]]), vecteurs_texte[c.index])[0])
    s_fam = np.where(c["family"].isin(profil["familles"]), 1.0, np.where(c["secondary_family"].isin(profil["familles"]), 0.5, 0.0))
    s_emo = normaliser(cosine_similarity(modele.encode([profil["emotions"]]), vecteurs_character[c.index])[0])
    s_sai = 0.7 * c[[f"season_{s}" for s in profil["saisons"]]].mean(axis=1) + 0.3 * c[[f"time_{m}" for m in profil["moments"]]].mean(axis=1)
    s_int = (1 - (c["intensity"] - profil["intensite"]).abs() / 4).fillna(0.5)
    c["score"] = 0.45 * s_sem + 0.20 * s_fam + 0.15 * s_emo + 0.12 * s_sai + 0.08 * s_int
    return c.sort_values("score", ascending=False).head(3)

MODELES = {"Word2Vec": recommander_w2v, "Sentence Transformers": recommander_st, "Hybride": recommander_hybride}





Fonction de comparaison

Pour chaque modèle, on affiche les 3 parfums les plus recommandés avec leur famille, leurs saisons et leur concentration.

In [46]:
def comparer(profil):
    criteres = ["Bonne famille", "Bonne saison", "Bonne intensité", "Sans note détestée"]
    resume = {}

    for nom, fonction in MODELES.items():
        print(f"\n=== {nom} ===")
        top = fonction(profil)
        if top is None:
            print("Aucun mot connu : pas de recommandation possible.")
            continue

        tableau = pd.DataFrame({
            "Parfum": top["name"],
            "Famille": top["family"],
            "Saisons": top["seasons"],
            "Concentration": top["concentration"],
            "Bonne famille": ["✓" if f in profil["familles"] or sf in profil["familles"] else "✗"
                              for f, sf in zip(top["family"], top["secondary_family"])],
            "Bonne saison": ["✓" if any(p[f"season_{s}"] == 1 for s in profil["saisons"]) else "✗"
                             for _, p in top.iterrows()],
            "Bonne intensité": ["✓" if abs(i - profil["intensite"]) <= 1 else "✗" for i in top["intensity"]],
            "Sans note détestée": ["✓" if not any(n in notes for n in profil["naime_pas"]) else "✗"
                                   for notes in top["notes"]],
        })
        display(tableau.reset_index(drop=True))

        resume[nom] = {c: f"{(tableau[c] == '✓').sum()}/3" for c in criteres}

    print("\n=== Récapitulatif ===")
    display(pd.DataFrame(resume).T)

Test 1 : profil classique

On test pour un utilisateur qui cherche un parfum boisé et épicé pour les soirées d'hiver, d'intensité moyenne et sans vanille

In [47]:
profil_1 = {
    "description": "I am looking for a warm and comforting woody perfume, slightly spicy, for winter evenings",
    "emotions": "cosy, reassuring",
    "naime_pas": ["vanilla"],
    "familles": ["Wood & Root", "Spice Route"],
    "saisons": ["winter"],
    "moments": ["evening"],
    "intensite": 3,
}

comparer(profil_1)



=== Word2Vec ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Sauvage Elixir,Green Leaf,spring; summer; autumn,elixir,✓,✗,✓,✓
1,Scandal pour Homme le Parfum,Velvet Sweet,autumn; winter; spring,parfum,✗,✓,✓,✓
2,Perfect Intense,Velvet Sweet,autumn; winter; spring,eau-de-parfum,✗,✓,✓,✓



=== Sentence Transformers ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,9 PM Night Out,Wood & Root,autumn; winter; spring,unknown,✓,✓,✗,✓
1,Toy Boy,Petal Garden,spring; summer; autumn,eau-de-parfum,✓,✗,✓,✓
2,Portrait Revival,Velvet Sweet,autumn; winter,unknown,✗,✓,✗,✓



=== Hybride ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Bucephalus No. XII,Spice Route,autumn; winter; spring,eau-de-parfum,✓,✓,✓,✓
1,Grey Vetiver,Spice Route,autumn; winter; spring,eau-de-parfum,✓,✓,✓,✓
2,The One for Men Eau de Parfum,Spice Route,autumn; winter; spring,eau-de-parfum,✓,✓,✓,✓



=== Récapitulatif ===


,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
Word2Vec,1/3,2/3,3/3,3/3
Sentence Transformers,2/3,2/3,1/3,3/3
Hybride,3/3,3/3,3/3,3/3


In [48]:
profil_2 = {
    "description": "I would like a fresh perfum but also not too sweet",
    "emotions": "cozy, woody",
    "naime_pas": ["lavender"],
    "familles": ["Amber & Resin"],
    "saisons": ["autumn"],
    "moments":["evening"],
    "intensite": 4,

}

comparer(profil_2)


=== Word2Vec ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Tuxedo,Spice Route,autumn; winter; spring,eau-de-parfum,✗,✓,✓,✓
1,Odyssey Toffee Coffee,Velvet Sweet,autumn; winter,eau-de-parfum,✗,✓,✓,✓
2,Oud Minérale,Oud Court,autumn; winter; spring,eau-de-parfum,✓,✓,✓,✓



=== Sentence Transformers ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Crystal Noir,Petal Garden,autumn; winter; spring,parfum,✗,✓,✓,✓
1,Sweet Oriental Dream,Velvet Sweet,autumn; winter,eau-de-parfum,✗,✓,✓,✓
2,Le Parfait Mon Amour,Velvet Sweet,autumn; winter; spring,unknown,✗,✓,✗,✓



=== Hybride ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Lune Feline,Amber & Resin,autumn; winter,eau-de-parfum,✓,✓,✓,✓
1,Il Padrino,Amber & Resin,autumn; winter; spring,eau-de-parfum,✓,✓,✓,✓
2,Oriental Lounge,Amber & Resin,autumn; winter; spring,eau-de-parfum,✓,✓,✓,✓



=== Récapitulatif ===


,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
Word2Vec,1/3,3/3,3/3,3/3
Sentence Transformers,0/3,3/3,2/3,3/3
Hybride,3/3,3/3,3/3,3/3


Test 2 
On va faire un test avec des mots inconnus

In [50]:
profil_3 = {
    "description": "Invigorating, revitalizing and effervescent, evoking an ocean vacation, energizing",
    "emotions": "uplifting, carefree, cheerful",
    "naime_pas":["oud"],
    "familles": ["Sunlit Citrus", "Salt & Mineral"],
    "saisons": ["summer"],
    "moments": ["day"],
    "intensite": 2,
}

mots = tokeniser(profil_3["description"])
print("Mots inconnus de Word2Vec :", [m for m in mots if m not in w2v.wv])

comparer(profil_3)

Mots inconnus de Word2Vec : ['invigorating', 'revitalizing', 'effervescent', 'evoking', 'ocean', 'vacation', 'energizing']

=== Word2Vec ===
Aucun mot connu : pas de recommandation possible.

=== Sentence Transformers ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Acqua di Giò Eau de Toilette,Petal Garden,spring; summer; autumn,eau-de-toilette,✗,✓,✓,✓
1,Acqua di Giò,Sunlit Citrus,spring; summer,eau-de-parfum,✓,✓,✓,✓
2,Gucci Flora Gorgeous Orchid,Velvet Sweet,autumn; winter; spring,eau-de-parfum,✗,✗,✓,✓



=== Hybride ===


,Parfum,Famille,Saisons,Concentration,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
0,Acqua di Giò,Sunlit Citrus,spring; summer,eau-de-parfum,✓,✓,✓,✓
1,L'Eau Majeure d'Issey,Sunlit Citrus,spring; summer,eau-de-toilette,✓,✓,✓,✓
2,Invictus Legend,Salt & Mineral,spring; summer; autumn,eau-de-parfum,✓,✓,✓,✓



=== Récapitulatif ===


,Bonne famille,Bonne saison,Bonne intensité,Sans note détestée
Sentence Transformers,1/3,2/3,3/3,3/3
Hybride,3/3,3/3,3/3,3/3


Conclusion

Le modèle hybride est nettement plus efficace pour retrouver le parfum le plus adapté à l'utilisateur.